# ❤️ Heart Disease Detection
### Using Machine Learning on the Cleveland Heart Disease Dataset
---
This notebook walks through:
1. Data loading & exploration
2. Preprocessing & feature engineering
3. Model training (Logistic Regression, Random Forest, XGBoost)
4. Evaluation & comparison
5. Feature importance
6. Interactive prediction

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, f1_score
)
from sklearn.pipeline import Pipeline
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('✅ All libraries imported successfully!')

## 2. Load Dataset

In [ ]:
# Load the Cleveland Heart Disease dataset
# Source: UCI Machine Learning Repository
url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/heart.csv'

try:
    df = pd.read_csv(url)
    print(f'✅ Dataset loaded from URL: {df.shape[0]} rows, {df.shape[1]} columns')
except Exception:
    # Fallback: generate synthetic data matching the Cleveland dataset structure
    print('⚠️  URL not reachable — generating synthetic Cleveland-style dataset...')
    from generate_data import generate_heart_dataset
    df = generate_heart_dataset(n=1000, random_state=42)
    print(f'✅ Synthetic dataset generated: {df.shape[0]} rows, {df.shape[1]} columns')

df.head()

In [ ]:
# Column descriptions
column_info = {
    'age':      'Age in years',
    'sex':      '1 = male, 0 = female',
    'cp':       'Chest pain type (0-3)',
    'trestbps': 'Resting blood pressure (mm Hg)',
    'chol':     'Serum cholesterol (mg/dl)',
    'fbs':      'Fasting blood sugar > 120 mg/dl (1=True)',
    'restecg':  'Resting ECG results (0-2)',
    'thalach':  'Maximum heart rate achieved',
    'exang':    'Exercise-induced angina (1=Yes)',
    'oldpeak':  'ST depression induced by exercise',
    'slope':    'Slope of peak exercise ST segment (0-2)',
    'ca':       'Number of major vessels (0-3)',
    'thal':     'Thalassemia (1=normal, 2=fixed defect, 3=reversible defect)',
    'target':   '1 = Heart Disease, 0 = No Heart Disease'
}

info_df = pd.DataFrame(list(column_info.items()), columns=['Feature', 'Description'])
print(info_df.to_string(index=False))

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Overview ===')
print(f'Shape: {df.shape}')
print(f'\nMissing Values:\n{df.isnull().sum()}')
print(f'\nTarget Distribution:\n{df["target"].value_counts()}')
print(f'\nClass Balance: {df["target"].value_counts(normalize=True).round(3).to_dict()}')
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Target distribution
target_counts = df['target'].value_counts()
axes[0, 0].pie(target_counts, labels=['No Disease', 'Disease'],
               autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
               startangle=90, shadow=True)
axes[0, 0].set_title('Target Class Distribution', fontsize=13, fontweight='bold')

# Age distribution by target
df[df['target'] == 0]['age'].hist(ax=axes[0, 1], alpha=0.7, color='#2ecc71', label='No Disease', bins=20)
df[df['target'] == 1]['age'].hist(ax=axes[0, 1], alpha=0.7, color='#e74c3c', label='Disease', bins=20)
axes[0, 1].set_title('Age Distribution by Target', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Age')
axes[0, 1].legend()

# Max heart rate by target
df.boxplot(column='thalach', by='target', ax=axes[1, 0], 
           boxprops=dict(color='steelblue'), medianprops=dict(color='red', linewidth=2))
axes[1, 0].set_title('Max Heart Rate by Target', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Target (0=No Disease, 1=Disease)')
plt.sca(axes[1, 0])
plt.title('Max Heart Rate by Target')

# Chest pain type
cp_target = df.groupby(['cp', 'target']).size().unstack(fill_value=0)
cp_target.plot(kind='bar', ax=axes[1, 1], color=['#2ecc71', '#e74c3c'], width=0.7)
axes[1, 1].set_title('Chest Pain Type vs Target', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Chest Pain Type')
axes[1, 1].legend(['No Disease', 'Disease'])
axes[1, 1].tick_params(axis='x', rotation=0)

plt.suptitle('Heart Disease – Exploratory Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved as eda_plots.png')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Preprocessing

In [ ]:
# Handle missing values
df = df.dropna()

# Ensure target is binary (UCI dataset sometimes has 0-4 scale)
df['target'] = (df['target'] > 0).astype(int)

# Features and target
X = df.drop('target', axis=1)
y = df['target']

# Train-test split (stratified to preserve class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train size : {X_train.shape[0]} samples')
print(f'Test  size : {X_test.shape[0]} samples')
print(f'Features   : {X.shape[1]}')
print(f'\nTrain class balance: {pd.Series(y_train).value_counts(normalize=True).round(3).to_dict()}')
print(f'Test  class balance: {pd.Series(y_test).value_counts(normalize=True).round(3).to_dict()}')

## 5. Train Multiple Models

In [ ]:
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=200, random_state=42),
    'SVM'                 : SVC(probability=True, kernel='rbf', random_state=42),
    'KNN'                 : KNeighborsClassifier(n_neighbors=7)
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred     = model.predict(X_test_scaled)
    y_prob     = model.predict_proba(X_test_scaled)[:, 1]
    cv_scores  = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')

    results[name] = {
        'model'       : model,
        'accuracy'    : accuracy_score(y_test, y_pred),
        'f1'          : f1_score(y_test, y_pred),
        'roc_auc'     : roc_auc_score(y_test, y_prob),
        'cv_mean'     : cv_scores.mean(),
        'cv_std'      : cv_scores.std(),
        'y_pred'      : y_pred,
        'y_prob'      : y_prob
    }
    print(f'✅ {name:<25} Acc={results[name]["accuracy"]:.3f}  F1={results[name]["f1"]:.3f}  AUC={results[name]["roc_auc"]:.3f}  CV={cv_scores.mean():.3f}±{cv_scores.std():.3f}')

## 6. Model Comparison

In [ ]:
# Summary table
summary = pd.DataFrame([
    {'Model': name, 'Accuracy': v['accuracy'], 'F1-Score': v['f1'],
     'ROC-AUC': v['roc_auc'], 'CV Accuracy': f"{v['cv_mean']:.3f} ± {v['cv_std']:.3f}"}
    for name, v in results.items()
]).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print('=== Model Performance Summary ===')
print(summary.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Metric bar chart
metrics   = ['accuracy', 'f1', 'roc_auc']
labels    = ['Accuracy', 'F1-Score', 'ROC-AUC']
model_names = list(results.keys())
x = np.arange(len(model_names))
width = 0.25
colors = ['#3498db', '#2ecc71', '#e74c3c']

for i, (metric, label, color) in enumerate(zip(metrics, labels, colors)):
    vals = [results[m][metric] for m in model_names]
    axes[0].bar(x + i * width, vals, width, label=label, color=color, alpha=0.85)

axes[0].set_xticks(x + width)
axes[0].set_xticklabels(model_names, rotation=20, ha='right', fontsize=9)
axes[0].set_ylim(0.5, 1.05)
axes[0].set_title('Model Performance Comparison', fontweight='bold')
axes[0].legend()
axes[0].set_ylabel('Score')

# ROC curves
for name, v in results.items():
    fpr, tpr, _ = roc_curve(y_test, v['y_prob'])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={v['roc_auc']:.3f})", linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[1].set_title('ROC Curves', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=8)

# Confusion matrix for best model
best_name  = max(results, key=lambda k: results[k]['roc_auc'])
best       = results[best_name]
cm = confusion_matrix(y_test, best['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
axes[2].set_title(f'Confusion Matrix – {best_name}', fontweight='bold')
axes[2].set_ylabel('Actual')
axes[2].set_xlabel('Predicted')

plt.suptitle('Heart Disease Detection – Model Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n🏆 Best model: {best_name}  (AUC={best["roc_auc"]:.3f})')

## 7. Best Model – Detailed Report

In [ ]:
print(f'=== Detailed Classification Report: {best_name} ===')
print(classification_report(y_test, best['y_pred'],
                             target_names=['No Disease', 'Heart Disease']))

## 8. Feature Importance

In [ ]:
# Use Random Forest for feature importance (always available)
rf_model  = results['Random Forest']['model']
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 7))
colors_bar = ['#e74c3c' if imp > importances.mean() else '#3498db' for imp in importances]
importances.plot(kind='barh', color=colors_bar)
plt.axvline(importances.mean(), color='gray', linestyle='--', linewidth=1.5, label='Mean importance')
plt.title('Feature Importances (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.legend()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 Most Important Features:')
print(importances.sort_values(ascending=False).head(5).round(4))

## 9. Save the Best Model

In [ ]:
best_model = best['model']

# Save model and scaler
joblib.dump(best_model, 'best_heart_model.pkl')
joblib.dump(scaler,     'scaler.pkl')

print(f'✅ Best model ({best_name}) saved to best_heart_model.pkl')
print('✅ Scaler saved to scaler.pkl')

## 10. Interactive Prediction

In [ ]:
def predict_heart_disease(age, sex, cp, trestbps, chol, fbs, restecg,
                           thalach, exang, oldpeak, slope, ca, thal):
    """
    Predict heart disease risk for a given patient.
    
    Parameters:
    -----------
    age      : int   – Age in years
    sex      : int   – 1=Male, 0=Female
    cp       : int   – Chest pain type (0=typical angina, 1=atypical, 2=non-anginal, 3=asymptomatic)
    trestbps : float – Resting blood pressure (mm Hg)
    chol     : float – Serum cholesterol (mg/dl)
    fbs      : int   – Fasting blood sugar > 120 mg/dl (1=True, 0=False)
    restecg  : int   – Resting ECG (0=normal, 1=ST-T abnormality, 2=LV hypertrophy)
    thalach  : float – Max heart rate achieved
    exang    : int   – Exercise-induced angina (1=Yes, 0=No)
    oldpeak  : float – ST depression induced by exercise
    slope    : int   – Slope of peak exercise ST segment (0=upsloping, 1=flat, 2=downsloping)
    ca       : int   – Number of major vessels colored by fluoroscopy (0-3)
    thal     : int   – Thalassemia (1=normal, 2=fixed defect, 3=reversible defect)
    """
    patient = np.array([[age, sex, cp, trestbps, chol, fbs, restecg,
                          thalach, exang, oldpeak, slope, ca, thal]])
    patient_scaled = scaler.transform(patient)
    prediction  = best_model.predict(patient_scaled)[0]
    probability = best_model.predict_proba(patient_scaled)[0][1]

    print('=' * 50)
    print('       HEART DISEASE RISK ASSESSMENT')
    print('=' * 50)
    print(f'  Model Used  : {best_name}')
    print(f'  Prediction  : {"⚠️  HEART DISEASE DETECTED" if prediction == 1 else "✅  NO HEART DISEASE"}')
    print(f'  Risk Prob.  : {probability:.1%}')
    print(f'  Risk Level  : {"🔴 HIGH" if probability > 0.7 else "🟡 MODERATE" if probability > 0.4 else "🟢 LOW"}')
    print('=' * 50)
    print('⚠️  This is a screening tool — always consult a physician.')
    return prediction, probability

# Example prediction – 55-year-old male with typical symptoms
predict_heart_disease(
    age=55, sex=1, cp=2, trestbps=140, chol=250,
    fbs=0, restecg=1, thalach=150, exang=1,
    oldpeak=2.3, slope=1, ca=1, thal=2
)

In [ ]:
# ✏️  Try your own patient — edit the values below!
predict_heart_disease(
    age=45,       # Age in years
    sex=0,        # 1=Male, 0=Female
    cp=1,         # Chest pain type 0-3
    trestbps=120, # Resting blood pressure
    chol=200,     # Cholesterol (mg/dl)
    fbs=0,        # Fasting blood sugar > 120? (1/0)
    restecg=0,    # Resting ECG result (0-2)
    thalach=170,  # Max heart rate achieved
    exang=0,      # Exercise-induced angina (1/0)
    oldpeak=0.5,  # ST depression
    slope=0,      # Slope (0-2)
    ca=0,         # Major vessels (0-3)
    thal=1        # Thalassemia (1-3)
)

---
## Summary

| Step | Details |
|------|----------|
| Dataset | Cleveland Heart Disease (UCI) – 13 clinical features |
| Models  | Logistic Regression, Random Forest, Gradient Boosting, SVM, KNN |
| Evaluation | Accuracy, F1-Score, ROC-AUC, 5-fold Cross-Validation |
| Outputs | Trained model (`.pkl`), scaler, and visualisation charts |

> ⚠️ **Disclaimer**: This model is built for educational purposes. Do not use it as a substitute for professional medical advice.
